# 📊 NumPy – Statistiques, agrégations et analyses (04)

---

## 🎯 Objectifs
- Calculer les **statistiques descriptives** : moyenne, médiane, écart-type, variance, percentiles
- Comprendre la différence entre **moyenne** et **médiane** et quand utiliser laquelle
- Agréger par **axe** (`axis=0` / `axis=1`)
- Comprendre et calculer la **corrélation** entre séries
- **Normaliser** des données : min-max et Z-score
- Calculer des **sommes cumulées** (`cumsum`)
- Appliquer tout ça sur des cas concrets : **météo**, **ventes**, **RH**, **Netflix**

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import numpy as np
print("NumPy version :", np.__version__)

---
## 📝 Partie 1 – Statistiques descriptives de base

### 🔎 Pourquoi les statistiques ?

Imaginez un tableau de 500 ventes journalières. Impossible de les lire une par une. Les **statistiques descriptives** permettent de **résumer** tout ça en quelques chiffres clés : combien en moyenne ? C'est stable ou chaotique ? Y a-t-il des journées exceptionnelles ?

### 🔎 Les indicateurs essentiels

| Indicateur | Fonction NumPy | Ce que ça dit concrètement |
|------------|---------------|----------------------------|
| Somme | `a.sum()` | Le total sur la période |
| Moyenne | `a.mean()` | La valeur "typique" (si les données sont régulières) |
| Médiane | `np.median(a)` | La valeur du milieu — pas influencée par les extrêmes |
| Minimum | `a.min()` | Le pire jour / la valeur la plus basse |
| Maximum | `a.max()` | Le meilleur jour / la valeur la plus haute |
| Étendue | `a.max() - a.min()` | L'amplitude totale des données |
| Écart-type | `a.std()` | À quel point les données sont irrégulières |
| Variance | `a.var()` | Écart-type au carré — même info, moins lisible |
| Percentile | `np.percentile(a, q)` | Valeur qui coupe la liste triée à q% du début |

### 🔎 Moyenne vs Médiane — laquelle choisir ?

La **moyenne** additionne tout et divise. Le problème : une seule valeur extrême suffit à la faire exploser.

La **médiane** prend simplement la valeur du milieu quand les données sont triées. Elle ignore les extrêmes.

```
5 salaires dans une entreprise, triés :
[2000, 2100, 2200, 2300, 15000]
                           ↑ PDG

Moyenne  = (2000+2100+2200+2300+15000) / 5 = 4720 €
           → Le salaire "moyen" est 4720 €... mais personne ne gagne ça !
           → Le PDG a tout faussé.

Médiane  = 2200 €  (c'est la valeur du milieu : 2 salaires en dessous, 2 au-dessus)
           → Beaucoup plus représentative de ce que gagnent vraiment les employés.
```

**Règle simple :**
- Données sans valeurs extrêmes (températures, tailles, délais...) → **Moyenne**
- Données avec valeurs extrêmes (salaires, prix immobilier, CA...) → **Médiane**

### 🔎 L'écart-type — les données sont-elles stables ?

L'écart-type mesure à quel point les valeurs sont **dispersées** autour de la moyenne. C'est la question : "est-ce que les données sont prévisibles, ou est-ce qu'elles sautent dans tous les sens ?"

**Exemple concret :** deux vendeurs ont la même moyenne de ventes sur 5 jours (10 pizzas/jour) :

```
Vendeur A : [10, 10, 10, 10, 10]  → std = 0
            Parfait pour planifier le stock. Toujours 10 pizzas, sans surprise.

Vendeur B : [2,  5,  10, 15, 18]  → std ≈ 5.5
            Même moyenne (10), mais imprévisible.
            Certains jours 2 pizzas, d'autres 18. Difficile de gérer le stock.
```

→ **std proche de 0** = données stables, faciles à anticiper  
→ **std élevé** = données irrégulières, risque plus difficile à gérer

**Variance vs écart-type :** la variance est l'écart-type au carré. Elle est utile pour les calculs internes des algorithmes, mais dans la pratique on préfère l'écart-type car il est exprimé dans **la même unité** que les données.

```
Ventes en pizzas : std = 5.5 pizzas  → "les ventes varient de ±5.5 pizzas autour de la moyenne"
                   var = 30.25       → pas en pizzas, difficile à lire
```

### 🔎 Les percentiles — situer une valeur dans la distribution

**L'idée** : on trie toutes les valeurs du plus petit au plus grand, et on regarde à quelle position se trouve un seuil donné.

**Exemple :** 10 salaires triés dans une entreprise :

```
[1500, 1800, 2000, 2200, 2400, 2600, 2900, 3200, 3800, 8000]
  10%   20%   30%   40%   50%   60%   70%   80%   90%  100%
              ↑                   ↑                   ↑
             Q1                  Q2                  Q3

Q1 (percentile 25) ≈ 2000 €
  → 1 employé sur 4 gagne MOINS de 2000 €

Q2 (percentile 50) ≈ 2500 €  =  médiane
  → La moitié des employés gagne MOINS de 2500 €

Q3 (percentile 75) ≈ 3200 €
  → 3 employés sur 4 gagnent MOINS de 3200 €
    (autrement dit : seul 1 sur 4 gagne plus)

P90 (percentile 90) ≈ 3800 €
  → Seuls 10% des employés gagnent plus de 3800 €
```

| Percentile | Nom | À retenir simplement |
|-----------|-----|---------------------|
| `np.percentile(a, 25)` | **Q1** | 1 valeur sur 4 est en dessous |
| `np.percentile(a, 50)` | **Q2** = médiane | 1 valeur sur 2 est en dessous |
| `np.percentile(a, 75)` | **Q3** | 3 valeurs sur 4 sont en dessous |
| `np.percentile(a, 90)` | **P90** | 9 valeurs sur 10 sont en dessous |

**L'IQR (Q3 - Q1)** mesure l'écart entre le premier et le troisième quartile. Il représente la dispersion de la moitié centrale des données, en ignorant les valeurs extrêmes. Le salaire du PDG à 8000 € ne fausse pas l'IQR.

```
IQR = 3200 - 2000 = 1200 €
→ La moitié centrale des salaires s'étale sur 1200 €
→ Le PDG à 8000 € ne change rien à ce résultat
```

**Cas d'usage :**
- RH : "Quel est le salaire du top 10% ?" → `np.percentile(salaires, 90)`
- E-commerce : "Quel est le panier des gros acheteurs (top 25%) ?" → `np.percentile(paniers, 75)`
- Qualité : "95% des pièces respectent quelle tolérance ?" → `np.percentile(mesures, 95)`

In [ ]:
import numpy as np

# Météo 🌡️ : températures max sur 7 jours
temperatures = np.array([18, 21, 22, 19, 24, 27, 25])
print("Températures :", temperatures)
print()
print(f"Moyenne    : {np.mean(temperatures):.1f} °C")
print(f"Médiane    : {np.median(temperatures):.1f} °C")
print(f"Min / Max  : {temperatures.min()} / {temperatures.max()} °C")
print(f"Étendue    : {temperatures.max() - temperatures.min()} °C")
print(f"Variance   : {np.var(temperatures):.2f}")
print(f"Écart-type : {np.std(temperatures):.2f} °C")
print()
print(f"Q1 (25e percentile) : {np.percentile(temperatures, 25)} °C")
print(f"Q2 (médiane)        : {np.percentile(temperatures, 50)} °C")
print(f"Q3 (75e percentile) : {np.percentile(temperatures, 75)} °C")
iqr = np.percentile(temperatures, 75) - np.percentile(temperatures, 25)
print(f"IQR (Q3 - Q1)       : {iqr} °C")

In [ ]:
import numpy as np

# Démonstration : impact d'une valeur aberrante
salaires_normaux = np.array([2000, 2100, 2200, 2300, 2400])
salaires_avec_ceo = np.array([2000, 2100, 2200, 2300, 15000])

print("=== Sans valeur aberrante ===")
print(f"Moyenne  : {salaires_normaux.mean():.0f} €")
print(f"Médiane  : {np.median(salaires_normaux):.0f} €")

print("\n=== Avec le salaire du CEO ===")
print(f"Moyenne  : {salaires_avec_ceo.mean():.0f} €  ← fortement biaisée")
print(f"Médiane  : {np.median(salaires_avec_ceo):.0f} €  ← stable, représentative")

👉 **Exercice 1** : Analyse des notes d'une classe.

Notes de 10 étudiants : `[8, 12, 14, 9, 17, 11, 15, 6, 13, 16]`

1. Calculez : moyenne, médiane, min, max, écart-type
2. Calculez les percentiles Q1, Q2, Q3 et l'IQR
3. Y a-t-il une note qui semble aberrante ? Comparez moyenne et médiane pour le savoir
4. Combien d'étudiants sont **au-dessus de la moyenne** ? (masque booléen)
5. Quel est le seuil de la note du **meilleur 20%** de la classe ? (`np.percentile` avec 80)

In [ ]:
import numpy as np

notes = np.array([8, 12, 14, 9, 17, 11, 15, 6, 13, 16])

# TODO 1 : statistiques de base
print(f"Moyenne    : {notes.mean():.1f}")
print(f"Médiane    : {np.median(notes):.1f}")
print(f"Min / Max  : {notes.min()} / {notes.max()}")
print(f"Écart-type : {np.std(notes):.2f}")

# TODO 2 : percentiles et IQR
q1  = ...
q2  = ...
q3  = ...
iqr = ...
print(f"\nQ1={q1} | Q2={q2} | Q3={q3} | IQR={iqr}")

# TODO 3 : comparer moyenne et médiane
print(f"\nMoyenne ({notes.mean():.1f}) vs Médiane ({np.median(notes):.1f}) : ", end="")
# ajoutez votre commentaire

# TODO 4 : étudiants au-dessus de la moyenne
print(f"Étudiants au-dessus de la moyenne : {(notes > notes.mean()).sum()}")

# TODO 5 : seuil du meilleur 20%
print(f"Seuil meilleur 20% : {np.percentile(notes, ...):.1f}")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np

notes = np.array([8, 12, 14, 9, 17, 11, 15, 6, 13, 16])

# 1
print(f"Moyenne    : {notes.mean():.1f}")       # 12.1
print(f"Médiane    : {np.median(notes):.1f}")   # 12.5
print(f"Min / Max  : {notes.min()} / {notes.max()}")
print(f"Écart-type : {np.std(notes):.2f}")      # 3.33

# 2
q1  = np.percentile(notes, 25)
q2  = np.percentile(notes, 50)
q3  = np.percentile(notes, 75)
iqr = q3 - q1
print(f"\nQ1={q1} | Q2={q2} | Q3={q3} | IQR={iqr}")

# 3 — moyenne ≈ médiane → peu de valeurs aberrantes ici
print(f"\nMoyenne ({notes.mean():.1f}) vs Médiane ({np.median(notes):.1f}) :")
print("Très proches → distribution équilibrée, pas de valeur très aberrante")

# 4
print(f"Étudiants au-dessus de la moyenne : {(notes > notes.mean()).sum()}")

# 5 — 80e percentile = seuil en dessous duquel se trouvent 80% des notes
print(f"Seuil meilleur 20% : {np.percentile(notes, 80):.1f}")
```
</details>

---
## 📝 Partie 2 – Statistiques par axe sur un tableau 2D

### 🔎 Pourquoi calculer par axe ?

Sur un tableau 2D, on ne veut pas toujours une seule statistique globale. On veut souvent savoir :
- **"Quelle saveur de pizza se vend le mieux en moyenne ?"** → statistique par colonne (par saveur)
- **"Quelle journée a été la plus forte ?"** → statistique par ligne (par jour)

C'est là qu'intervient `axis` :

```
Tableau ventes : 7 jours × 3 saveurs

              Margherita  Reine  Végé
Jour 0   [     12         15     10  ]  ──► axis=1 : moyenne de CE jour
Jour 1   [     14         16     11  ]  ──► axis=1 : moyenne de CE jour
Jour 2   [     13         14      9  ]  ──► axis=1 : moyenne de CE jour
  ...             ↓           ↓      ↓
               axis=0      axis=0  axis=0
          moyenne de     moyenne  moyenne
          Margherita     Reine    Végé
          sur 7 jours    sur 7j   sur 7j
```

| `axis` | Question à laquelle ça répond | Résultat |
|--------|------------------------------|----------|
| `axis=0` | "Quelle est la stat de chaque **saveur** sur toute la période ?" | 1 valeur par colonne |
| `axis=1` | "Quelle est la stat de chaque **journée** toutes saveurs confondues ?" | 1 valeur par ligne |
| *(aucun)* | "Quelle est la stat globale de tout le tableau ?" | 1 seule valeur |

Toutes les fonctions statistiques acceptent `axis` de la même façon :
```python
ventes.mean(axis=0)            # moyenne par saveur (par colonne)
ventes.std(axis=1)             # écart-type par jour (par ligne)
np.median(ventes, axis=0)      # médiane par saveur
np.percentile(ventes, 75, axis=1)  # Q3 par jour
```

In [ ]:
import numpy as np

# Ventes de pizzas 🍕 : 7 jours × 3 saveurs
ventes = np.array([[12, 15, 10],
                   [14, 16, 11],
                   [13, 14,  9],
                   [18, 20, 12],
                   [17, 18, 14],
                   [19, 21, 15],
                   [16, 19, 13]])

saveurs = ["Margherita", "Reine", "Végétarienne"]

print("=== Statistiques par saveur (axis=0) ===")
print(f"Moyenne   : {np.mean(ventes, axis=0)}")
print(f"Médiane   : {np.median(ventes, axis=0)}")
print(f"Std       : {np.std(ventes, axis=0).round(2)}")
print(f"Total     : {ventes.sum(axis=0)}")

print()
print("=== Statistiques par jour (axis=1) ===")
print(f"Total     : {ventes.sum(axis=1)}")
print(f"Moyenne   : {ventes.mean(axis=1).round(1)}")
print(f"Std       : {ventes.std(axis=1).round(2)}")

print()
# Quelle saveur a les ventes les plus stables (écart-type le plus faible) ?
std_par_saveur = np.std(ventes, axis=0)
saveur_stable  = np.argmin(std_par_saveur)
print(f"Saveur la plus stable : {saveurs[saveur_stable]} (std={std_par_saveur[saveur_stable]:.2f})")

# Quel jour a la plus grande variabilité entre saveurs ?
std_par_jour = np.std(ventes, axis=1)
jour_variable = np.argmax(std_par_jour)
print(f"Jour le plus variable : jour {jour_variable} (std={std_par_jour[jour_variable]:.2f})")

👉 **Exercice 2** : Analyse des résultats d'une chaîne hôtelière.

Taux d'occupation (%) de 4 hôtels sur 6 mois :
```python
occupation = np.array([[72, 85, 60, 91],
                       [68, 80, 55, 88],
                       [75, 90, 65, 95],
                       [80, 88, 70, 92],
                       [85, 92, 75, 97],
                       [70, 83, 58, 89]])
hotels = ["Paris", "Lyon", "Marseille", "Nice"]
```

1. Calculez le taux d'occupation **moyen par hôtel** sur les 6 mois
2. Calculez le taux d'occupation **moyen par mois** tous hôtels confondus
3. Quel hôtel a le taux moyen **le plus élevé** ? Lequel est le **plus stable** (std la plus faible) ?
4. Quel mois a été **le meilleur** pour l'ensemble de la chaîne ?
5. Calculez le **Q1 et Q3 par hôtel** — y a-t-il un hôtel très irrégulier ?

In [ ]:
import numpy as np

occupation = np.array([[72, 85, 60, 91],
                       [68, 80, 55, 88],
                       [75, 90, 65, 95],
                       [80, 88, 70, 92],
                       [85, 92, 75, 97],
                       [70, 83, 58, 89]])
hotels = ["Paris", "Lyon", "Marseille", "Nice"]

# TODO 1 : moyenne par hôtel
moy_hotel = ...
for i, h in enumerate(hotels):
    print(f"  {h} : {moy_hotel[i]:.1f}%")

# TODO 2 : moyenne par mois
moy_mois = ...
print("\nMoyenne par mois :", moy_mois.round(1))

# TODO 3 : meilleur hôtel et plus stable
print("\nHôtel avec meilleur taux  :", hotels[...])
std_hotel = ...
print("Hôtel le plus stable      :", hotels[np.argmin(std_hotel)])

# TODO 4 : meilleur mois
print("Meilleur mois (index)     :", ...)

# TODO 5 : Q1 et Q3 par hôtel
q1 = np.percentile(occupation, 25, axis=0)
q3 = np.percentile(occupation, 75, axis=0)
iqr = ...
print("\nIQR par hôtel :", iqr)
print("Hôtel le plus irrégulier :", hotels[np.argmax(iqr)])

<details>
<summary>💡 Correction</summary>

```python
import numpy as np

occupation = np.array([[72, 85, 60, 91],
                       [68, 80, 55, 88],
                       [75, 90, 65, 95],
                       [80, 88, 70, 92],
                       [85, 92, 75, 97],
                       [70, 83, 58, 89]])
hotels = ["Paris", "Lyon", "Marseille", "Nice"]

# 1 — axis=0 : on agrège les 6 mois → 1 valeur par hôtel
moy_hotel = occupation.mean(axis=0)
for i, h in enumerate(hotels):
    print(f"  {h} : {moy_hotel[i]:.1f}%")

# 2 — axis=1 : on agrège les 4 hôtels → 1 valeur par mois
moy_mois = occupation.mean(axis=1)
print("\nMoyenne par mois :", moy_mois.round(1))

# 3
print("\nHôtel avec meilleur taux  :", hotels[np.argmax(moy_hotel)])
std_hotel = occupation.std(axis=0)
print("Hôtel le plus stable      :", hotels[np.argmin(std_hotel)])

# 4
print("Meilleur mois (index)     :", np.argmax(moy_mois))

# 5
q1  = np.percentile(occupation, 25, axis=0)
q3  = np.percentile(occupation, 75, axis=0)
iqr = q3 - q1
print("\nIQR par hôtel :", iqr)
print("Hôtel le plus irrégulier :", hotels[np.argmax(iqr)])
```
</details>

---
## 📝 Partie 3 – Corrélation

### 🔎 Qu'est-ce que la corrélation ?

La corrélation répond à une question simple : **est-ce que deux choses évoluent ensemble ?**

**Exemples du quotidien :**
- Quand il fait **chaud**, on vend **plus de glaces** → les deux montent ensemble → **corrélation positive**
- Quand il fait **chaud**, on vend **moins de manteaux** → l'un monte, l'autre descend → **corrélation négative**
- La **météo** et le **cours de la Bourse** → aucun lien → **pas de corrélation**

En data science, on quantifie ça avec un **coefficient de corrélation** compris entre -1 et +1 :

```
                    -1          0          +1
                     |          |           |
  relation inverse   |  aucun   |  relation |
  parfaite           |  lien    |  parfaite |
  (A↑ → B↓)         |          |  (A↑ → B↑)|
```

**Exemple chiffré :** températures et ventes de glaces sur 5 jours

```
Jour        :  Lun   Mar   Mer   Jeu   Ven
Température :  15    20    28    32    25   °C
Glaces vend.:  30    50    85   110    70   unités

→ Quand la température monte, les ventes montent aussi.
→ Corrélation ≈ +0.99  (quasi parfaite, très forte relation positive)
```

**Contre-exemple :** températures et ventes de parapluies

```
Jour          :  Lun   Mar   Mer   Jeu   Ven
Température   :  15    20    28    32    25   °C  (identique)
Parapluies    :  80    60    20    10    30   unités

→ Quand la température monte, les ventes descendent.
→ Corrélation ≈ -0.97  (forte relation négative)
```

**Interprétation pratique :**

| Valeur absolue | Interprétation | Exemple |
|---------------|----------------|--------|
| 0.9 – 1.0 | Très forte | Température ↔ ventes de glaces |
| 0.7 – 0.9 | Forte | Budget pub ↔ nombre de clics |
| 0.5 – 0.7 | Modérée | Ancienneté ↔ salaire |
| 0.3 – 0.5 | Faible | Météo ↔ humeur des clients |
| 0.0 – 0.3 | Très faible ou nulle | Pointure de chaussure ↔ salaire |

> ⚠️ **Corrélation ≠ causalité** : deux choses peuvent évoluer ensemble sans que l'une soit la **cause** de l'autre. Exemple célèbre : la consommation de glaces et les noyades sont corrélées positivement — mais c'est la chaleur qui explique les deux, pas les glaces qui provoquent les noyades !

### 🔎 `np.corrcoef` — matrice de corrélation

```python
# Deux séries de 5 valeurs
temperature = np.array([15, 20, 28, 32, 25])
glaces      = np.array([30, 50, 85, 110, 70])

np.corrcoef(temperature, glaces)
# → [[1.    0.99]   ← corrélation de temperature avec elle-même (toujours 1)
#    [0.99  1.  ]]  ← corrélation de glaces avec elle-même (toujours 1)
#      ↑
#   corrélation entre temperature et glaces = 0.99 → très forte relation positive
```

Pour un tableau 2D avec **plusieurs séries** :
```python
# minutes : shape (7, 3) — 7 jours, 3 séries Netflix
# On veut la corrélation ENTRE SÉRIES → chaque série doit être une LIGNE
np.corrcoef(minutes.T)   # .T transpose : les colonnes deviennent des lignes
# → matrice 3×3 : chaque case [i,j] = corrélation entre série i et série j
```

In [ ]:
import numpy as np

# Netflix 🎬 : minutes vues — 7 jours × 3 séries
np.random.seed(42)
minutes = np.random.randint(20, 60, (7, 3))
series  = ["Série A", "Série B", "Série C"]
print("Minutes vues :\n", minutes)

# Corrélation entre les séries
# minutes.T : shape (3, 7) → chaque LIGNE est une série
corr = np.corrcoef(minutes.T)
print("\nMatrice de corrélation (séries × séries) :")
print(np.round(corr, 2))

print()

# Lire la matrice
for i in range(len(series)):
    for j in range(i+1, len(series)):
        val = corr[i, j]
        if abs(val) >= 0.7:
            force = "forte"
        elif abs(val) >= 0.5:
            force = "modérée"
        else:
            force = "faible"
        signe = "positive" if val > 0 else "négative"
        print(f"{series[i]} ↔ {series[j]} : {val:.2f} → corrélation {force} {signe}")

print()

# Exemple clair de corrélation positive et négative
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pos = np.array([2.1, 3.9, 6.2, 8.0, 9.8])   # fortement corrélé positivement
y_neg = np.array([9.5, 8.1, 5.9, 4.2, 2.0])   # fortement corrélé négativement
y_nul = np.array([3.0, 7.0, 2.0, 8.0, 1.0])   # pas de corrélation

print("Corrélation x ↔ y_pos :", round(np.corrcoef(x, y_pos)[0, 1], 3))
print("Corrélation x ↔ y_neg :", round(np.corrcoef(x, y_neg)[0, 1], 3))
print("Corrélation x ↔ y_nul :", round(np.corrcoef(x, y_nul)[0, 1], 3))

👉 **Exercice 3** : Marketing digital — analyse des corrélations.

Sur 10 jours, on mesure : budget pub (€), clics, conversions, chiffre d'affaires (€).
```python
np.random.seed(5)
budget      = np.random.randint(100, 500, 10).astype(float)
clics       = budget * 2.5 + np.random.randn(10) * 50
conversions = clics * 0.03 + np.random.randn(10) * 2
ca          = conversions * 45 + np.random.randn(10) * 30
```

1. Créez une matrice `donnees` de shape `(4, 10)` en empilant les 4 séries (chaque série = une ligne)
2. Calculez la matrice de corrélation avec `np.corrcoef`
3. Affichez la corrélation budget ↔ CA et clics ↔ conversions
4. Quelle paire de variables est la plus corrélée ?

In [ ]:
import numpy as np
np.random.seed(5)

budget      = np.random.randint(100, 500, 10).astype(float)
clics       = budget * 2.5 + np.random.randn(10) * 50
conversions = clics * 0.03 + np.random.randn(10) * 2
ca          = conversions * 45 + np.random.randn(10) * 30
labels      = ["Budget", "Clics", "Conversions", "CA"]

# TODO 1 : empiler les 4 séries en shape (4, 10)
# Indice : np.array([serie1, serie2, ...]) crée une matrice
donnees = np.array([budget, clics, conversions, ca])
print("Shape données :", donnees.shape)   # doit être (4, 10)

# TODO 2 : matrice de corrélation
corr = ...
print("\nMatrice de corrélation :")
print(np.round(corr, 2))

# TODO 3 : corrélations spécifiques
# corr[i, j] = corrélation entre la série i et la série j
print(f"\nBudget ↔ CA            : {corr[0, 3]:.3f}")
print(f"Clics  ↔ Conversions   : {corr[...]}")

# TODO 4 : paire la plus corrélée (hors diagonale)
# La diagonale contient toujours 1 (corrélation d'une série avec elle-même)
# On la met à 0 pour l'ignorer, puis on cherche le max
corr_abs = np.abs(corr.copy())
np.fill_diagonal(corr_abs, 0)   # ignorer la diagonale
i, j = np.where(corr_abs == corr_abs.max())
print(f"Paire la plus corrélée : {labels[i[0]]} ↔ {labels[j[0]]} ({corr[i[0], j[0]]:.3f})")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np
np.random.seed(5)

budget      = np.random.randint(100, 500, 10).astype(float)
clics       = budget * 2.5 + np.random.randn(10) * 50
conversions = clics * 0.03 + np.random.randn(10) * 2
ca          = conversions * 45 + np.random.randn(10) * 30
labels      = ["Budget", "Clics", "Conversions", "CA"]

# 1 — np.array() d'une liste de vecteurs → matrice (4, 10)
donnees = np.array([budget, clics, conversions, ca])
print("Shape données :", donnees.shape)

# 2
corr = np.corrcoef(donnees)
print("\nMatrice de corrélation :")
print(np.round(corr, 2))

# 3
print(f"\nBudget ↔ CA            : {corr[0, 3]:.3f}")
print(f"Clics  ↔ Conversions   : {corr[1, 2]:.3f}")

# 4 — np.where retourne les indices des positions où la condition est vraie
corr_abs = np.abs(corr.copy())
np.fill_diagonal(corr_abs, 0)
i, j = np.where(corr_abs == corr_abs.max())
print(f"Paire la plus corrélée : {labels[i[0]]} ↔ {labels[j[0]]} ({corr[i[0], j[0]]:.3f})")
```
</details>

---
## 📝 Partie 4 – Normalisation des données

### 🔎 Pourquoi normaliser ?

Imaginez que vous voulez comparer l'évolution des ventes (entre 500 et 50 000 €) avec l'évolution de la température (entre 0 et 35°C). Impossible de les mettre sur le même graphique ou de les comparer directement — les unités et les échelles sont trop différentes.

La normalisation **ramène toutes les séries sur une échelle commune** pour pouvoir les comparer sur un pied d'égalité.

### 🔎 Normalisation Min-Max — tout ramener entre 0 et 1

**Principe :** la valeur la plus basse devient 0, la plus haute devient 1, les autres se placent proportionnellement entre les deux.

```
Formule : x_norm = (x - x_min) / (x_max - x_min)

Exemple — ventes sur 5 jours : [100, 200, 400, 800, 1500]

  100  → (100  - 100) / (1500 - 100) = 0/1400  = 0.00  ← le pire jour
  200  → (200  - 100) / (1500 - 100) = 100/1400 = 0.07
  400  → (400  - 100) / (1500 - 100) = 300/1400 = 0.21
  800  → (800  - 100) / (1500 - 100) = 700/1400 = 0.50
 1500  → (1500 - 100) / (1500 - 100) = 1400/1400 = 1.00 ← le meilleur jour
```

**Avantage** : toujours entre 0 et 1, facile à comparer.  
**Inconvénient** : si un seul chiffre est aberrant (ex: une vente exceptionnelle à 100 000 €), tout le reste se retrouve écrasé vers 0.

### 🔎 Z-score — se situer par rapport à la moyenne

**Principe :** on exprime chaque valeur en **"à combien d'écarts-types est-elle de la moyenne ?"**. C'est une façon de savoir si une valeur est normale ou exceptionnelle.

```
Formule : x_z = (x - moyenne) / écart_type

Exemple — salaires : moyenne = 2500 €, écart-type = 500 €

  2500 € → (2500 - 2500) / 500 =  0.0  → pile dans la moyenne, normal
  3000 € → (3000 - 2500) / 500 =  1.0  → 1 cran au-dessus de la moyenne
  1500 € → (1500 - 2500) / 500 = -2.0  → 2 crans en dessous, assez bas
  5000 € → (5000 - 2500) / 500 =  5.0  → très loin, probablement un cadre ou PDG
```

**Après Z-score :**
- La moyenne de la série = **0**
- L'écart-type = **1**
- Valeur positive → au-dessus de la moyenne d'origine
- Valeur négative → en dessous de la moyenne d'origine
- Z > 2 ou Z < -2 → valeur atypique, mérite d'être examinée

**Avantage** : robuste, interprétable, résistant aux outliers.  
**Inconvénient** : pas borné entre 0 et 1.

### 🔎 Quand utiliser laquelle ?

| Situation | Recommandation |
|-----------|---------------|
| Données avec échelle fixe connue (notes /20, %) | Min-Max |
| Données avec valeurs aberrantes possibles | Z-score |
| Algorithmes de ML (KNN, SVM, réseaux de neurones) | Min-Max ou Z-score |
| Détecter des valeurs suspectes | Z-score (Z > 2 = suspect) |
| Comparer deux séries d'unités différentes | Les deux fonctionnent |

In [ ]:
import numpy as np

# Comparer des grandeurs hétérogènes
ventes = np.array([100, 200, 400, 800, 1500], dtype=float)
temp   = np.array([  5,  12,  18,  25,   30], dtype=float)

print("=== Normalisation Min-Max (0–1) ===")
def minmax(x):
    return (x - x.min()) / (x.max() - x.min())

ventes_mm = minmax(ventes)
temp_mm   = minmax(temp)
print("Ventes norm. :", np.round(ventes_mm, 3))
print("Temp.  norm. :", np.round(temp_mm, 3))

print()
print("=== Standardisation Z-score ===")
def zscore(x):
    return (x - x.mean()) / x.std()

ventes_z = zscore(ventes)
temp_z   = zscore(temp)
print("Ventes Z-score :", np.round(ventes_z, 3))
print("Temp.  Z-score :", np.round(temp_z, 3))
print("Vérification : mean ≈ 0 ?", round(ventes_z.mean(), 10))
print("Vérification : std  = 1 ?", round(ventes_z.std(), 10))

print()
print("=== Impact d'un outlier ===")
avec_outlier = np.array([10, 12, 11, 13, 10, 12, 100])  # 100 = outlier
print("Min-Max :", np.round(minmax(avec_outlier), 3))
print("Z-score :", np.round(zscore(avec_outlier), 3))
# → Min-Max : toutes les normales sont entre 0 et 0.03, l'outlier écrase tout
# → Z-score : les normales ont des scores proches, l'outlier ressort clairement

👉 **Exercice 4** : Comparaison de canaux marketing.

Sur 7 jours, on suit le trafic (visiteurs) et le taux de conversion (%) :
```python
trafic     = np.array([1200, 1400, 1500, 800, 1700, 1650, 1750], dtype=float)
conversion = np.array([0.6,  0.8,  1.0, 1.4,  2.0,  1.7,  2.3])
```

1. Normalisez les deux séries en **Min-Max**
2. Normalisez les deux séries en **Z-score**
3. Quel jour a le Z-score de trafic **le plus négatif** ? Que signifie-t-il ?
4. Calculez la corrélation entre trafic et conversion — sont-ils liés ?

In [ ]:
import numpy as np

trafic     = np.array([1200, 1400, 1500,  800, 1700, 1650, 1750], dtype=float)
conversion = np.array([0.6,  0.8,  1.0,  1.4,  2.0,  1.7,  2.3])

# TODO 1 : normalisation Min-Max
trafic_mm = ...
conv_mm   = ...
print("Trafic Min-Max     :", np.round(trafic_mm, 3))
print("Conversion Min-Max :", np.round(conv_mm, 3))

# TODO 2 : Z-score
trafic_z = ...
conv_z   = ...
print("\nTrafic Z-score     :", np.round(trafic_z, 2))
print("Conversion Z-score :", np.round(conv_z, 2))

# TODO 3 : jour avec Z-score trafic le plus négatif
jour_creux = ...
print(f"\nJour le plus creux (Z-score trafic) : jour {jour_creux}")
print("Interprétation : ce jour est", abs(trafic_z[jour_creux]).round(2), "écarts-types EN DESSOUS de la moyenne")

# TODO 4 : corrélation trafic ↔ conversion
r = ...
print(f"\nCorrélation trafic ↔ conversion : {r:.3f}")

<details>
<summary>💡 Correction</summary>

```python
import numpy as np

trafic     = np.array([1200, 1400, 1500,  800, 1700, 1650, 1750], dtype=float)
conversion = np.array([0.6,  0.8,  1.0,  1.4,  2.0,  1.7,  2.3])

# 1 — Min-Max
trafic_mm = (trafic - trafic.min()) / (trafic.max() - trafic.min())
conv_mm   = (conversion - conversion.min()) / (conversion.max() - conversion.min())
print("Trafic Min-Max     :", np.round(trafic_mm, 3))
print("Conversion Min-Max :", np.round(conv_mm, 3))

# 2 — Z-score
trafic_z = (trafic - trafic.mean()) / trafic.std()
conv_z   = (conversion - conversion.mean()) / conversion.std()
print("\nTrafic Z-score     :", np.round(trafic_z, 2))
print("Conversion Z-score :", np.round(conv_z, 2))

# 3
jour_creux = np.argmin(trafic_z)
print(f"\nJour le plus creux : jour {jour_creux}")
print("Z-score :", trafic_z[jour_creux].round(2),
      "→ ce jour est", abs(trafic_z[jour_creux]).round(2),
      "écarts-types EN DESSOUS de la moyenne")

# 4
r = np.corrcoef(trafic, conversion)[0, 1]
print(f"\nCorrélation trafic ↔ conversion : {r:.3f}")
# → corrélation positive : plus le trafic est élevé, meilleure la conversion
```
</details>

---
## 📝 Partie 5 – Somme cumulée (`cumsum`)

### 🔎 Qu'est-ce que `cumsum` ?

`np.cumsum` calcule la **somme cumulée** : chaque élément est la somme de tous les éléments précédents plus lui-même.

```python
ventes = np.array([10, 15, 12, 20, 18])
np.cumsum(ventes)
# → [10, 25, 37, 57, 75]
#    ↑   ↑   ↑   ↑   ↑
#    10  10  10  10  10
#        +15 +15 +15 +15
#            +12 +12 +12
#                +20 +20
#                    +18
```

**Cas d'usage** :
- Suivi du **chiffre d'affaires cumulé** dans le temps
- Courbe de **progression** (objectif atteint à quel %)
- **Calcul de pourcentage cumulé** pour les diagrammes de Pareto

La variante `np.cumprod` calcule le **produit cumulé** (utile pour les rendements financiers composés).

In [ ]:
import numpy as np

# Ventes journalières sur 2 semaines
ventes = np.array([120, 145, 98, 160, 175, 140, 190,
                   130, 155, 110, 170, 185, 150, 200])

ca_cumule = np.cumsum(ventes)
print("Ventes journalières :", ventes)
print("CA cumulé           :", ca_cumule)

# Objectif mensuel : 2000 €
objectif = 2000
print(f"\nObjectif : {objectif} €")
print(f"CA total  : {ca_cumule[-1]} €")
print(f"Objectif atteint à {ca_cumule[-1]/objectif*100:.1f}%")

# Quel jour dépasse-t-on 50% de l'objectif ?
seuil_50 = objectif * 0.5
jour_50  = np.where(ca_cumule >= seuil_50)[0][0]
print(f"50% de l'objectif franchi au jour {jour_50} "
      f"(CA cumulé : {ca_cumule[jour_50]} €)")

print()
# Pourcentage cumulé des ventes (pour Pareto)
ventes_triees = np.sort(ventes)[::-1]   # tri décroissant
pct_cumule = np.cumsum(ventes_triees) / ventes.sum() * 100
print("Ventes triées décroissant :", ventes_triees)
print("% cumulé                  :", np.round(pct_cumule, 1))

👉 **Exercice 5** : Suivi de budget projet.

Dépenses mensuelles (€) sur 12 mois :
```python
depenses = np.array([1500, 2200, 1800, 3000, 2500, 1900,
                     2100, 2800, 2000, 1700, 2400, 3200])
budget_annuel = 30000
```

1. Calculez les dépenses cumulées mois par mois
2. Calculez le **budget restant** mois par mois (`budget_annuel - cumsum`)
3. À quel mois dépasse-t-on **50%** du budget ?
4. À quel mois dépasse-t-on **80%** du budget ?
5. Le budget est-il dépassé en fin d'année ?

In [ ]:
import numpy as np

depenses = np.array([1500, 2200, 1800, 3000, 2500, 1900,
                     2100, 2800, 2000, 1700, 2400, 3200])
budget_annuel = 30000
mois = ["Jan","Fév","Mar","Avr","Mai","Jun",
        "Jul","Aoû","Sep","Oct","Nov","Déc"]

# TODO 1 : dépenses cumulées
cumul = ...
print("Dépenses cumulées :")
for i, m in enumerate(mois):
    print(f"  {m} : {cumul[i]:>6} €")

# TODO 2 : budget restant
restant = ...
print("\nBudget restant en décembre :", restant[-1], "€")

# TODO 3 : mois où on dépasse 50% du budget
seuil_50 = budget_annuel * 0.5
mois_50  = ...
print(f"\n50% du budget franchi en : {mois[mois_50]}")

# TODO 4 : mois où on dépasse 80% du budget
mois_80 = ...
print(f"80% du budget franchi en : {mois[mois_80]}")

# TODO 5 : budget dépassé ?
print(f"\nTotal dépenses : {cumul[-1]} €")
print(f"Budget annuel  : {budget_annuel} €")
print("Budget dépassé ?", ...)

<details>
<summary>💡 Correction</summary>

```python
import numpy as np

depenses = np.array([1500, 2200, 1800, 3000, 2500, 1900,
                     2100, 2800, 2000, 1700, 2400, 3200])
budget_annuel = 30000
mois = ["Jan","Fév","Mar","Avr","Mai","Jun",
        "Jul","Aoû","Sep","Oct","Nov","Déc"]

# 1
cumul = np.cumsum(depenses)
print("Dépenses cumulées :")
for i, m in enumerate(mois):
    print(f"  {m} : {cumul[i]:>6} €")

# 2
restant = budget_annuel - cumul
print("\nBudget restant en décembre :", restant[-1], "€")

# 3
seuil_50 = budget_annuel * 0.5
mois_50  = np.where(cumul >= seuil_50)[0][0]
print(f"\n50% du budget franchi en : {mois[mois_50]}")

# 4
seuil_80 = budget_annuel * 0.8
mois_80  = np.where(cumul >= seuil_80)[0][0]
print(f"80% du budget franchi en : {mois[mois_80]}")

# 5
print(f"\nTotal dépenses : {cumul[-1]} €")
print(f"Budget annuel  : {budget_annuel} €")
print("Budget dépassé ?", cumul[-1] > budget_annuel)
# → True si dépassé, False sinon
```
</details>

---
## ✅ Récapitulatif

| Concept | Fonction | Ce qu'il faut retenir |
|---------|----------|-----------------------|
| Moyenne | `a.mean()` | Sensible aux outliers |
| Médiane | `np.median(a)` | Robuste aux outliers — à préférer pour les salaires, prix |
| Écart-type | `a.std()` | Mesure la dispersion — faible = stable, élevé = irrégulier |
| Variance | `a.var()` | Écart-type au carré — même info mais moins lisible |
| Percentile | `np.percentile(a, q)` | P25 = 1 valeur sur 4 en dessous, P50 = moitié, P75 = 3 sur 4 |
| IQR | `Q3 - Q1` | Dispersion de la moitié centrale — insensible aux valeurs extrêmes |
| Par axe | `a.mean(axis=0/1)` | `axis=0` = par colonne, `axis=1` = par ligne |
| Corrélation | `np.corrcoef(a, b)` | Entre -1 et +1 — ne signifie pas causalité |
| Normalisation Min-Max | `(x-min)/(max-min)` | Ramène entre 0 et 1 — sensible aux outliers |
| Z-score | `(x-mean)/std` | Distance à la moyenne en nb d'écarts-types — Z=0 : dans la moyenne |
| Somme cumulée | `np.cumsum(a)` | Suivi de progression dans le temps |

---
📘 **Prochain notebook → `05` : Manipulation d'images avec NumPy**